# SD3.5 LoRA Training — Kaggle Notebook

Orchestrates the full pipeline: `raw datasets → build release → validate → train → export → load-back smoke test`.

**All logic lives in Python modules. This notebook only calls them.**  
Fails fast at Cell 05, 07, or 08 if any contract is not met.

In [ ]:
# Cell 00: Verify git commit and environment
import subprocess, sys

result = subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True)
GIT_SHA = result.stdout.strip() if result.returncode == 0 else 'unknown'
print(f'Git SHA: {GIT_SHA}')
print(f'Python: {sys.version}')

In [ ]:
# Cell 01: GPU preflight — must show a CUDA GPU before continuing
import torch

assert torch.cuda.is_available(), 'No CUDA GPU detected. Stop and attach a GPU accelerator.'

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'GPU: {gpu_name}')
print(f'VRAM: {vram_gb:.1f} GB')
print(f'CUDA: {torch.version.cuda}')
print(f'PyTorch: {torch.__version__}')

if vram_gb < 14:
    print(f'WARNING: VRAM {vram_gb:.1f} GB < 14 GB recommended for fp16 batch=1 at 512px')

In [ ]:
# Cell 02: Install pinned dependencies
# Pin versions to avoid silent API breakage in the Diffusers training script.
!pip install -q \
    'diffusers==0.31.0' \
    'transformers>=4.44.0' \
    'accelerate>=0.33.0' \
    'datasets>=2.20.0' \
    'safetensors>=0.4.3' \
    'bitsandbytes>=0.43.0' \
    'peft>=0.12.0' \
    'pillow>=10.0.0' \
    'imagehash>=4.3.1' \
    'pandas>=2.0.0' \
    'pyarrow>=14.0.0' \
    'tqdm'

import diffusers, datasets, accelerate, safetensors
print(f'diffusers: {diffusers.__version__}')
print(f'datasets:  {datasets.__version__}')
print(f'accelerate:{accelerate.__version__}')
print(f'safetensors:{safetensors.__version__}')

In [ ]:
# Cell 03: Verify Kaggle mounts and source inventory
from pathlib import Path
import sys

# Add project root to path
sys.path.insert(0, '/kaggle/working')

SOURCES = {
    'citypersons':      '/kaggle/input/citypersons-canonical',
    'mot17_02':         '/kaggle/input/mot17-02-fcrnn',
    'human_detection':  '/kaggle/input/human-detection-dataset',
    'sd35_model':       '/kaggle/input/stable-diffusion-3-5-medium',
}

all_ok = True
for name, path in SOURCES.items():
    exists = Path(path).exists()
    status = '✓' if exists else '✗ MISSING'
    print(f'  {status}  {name}: {path}')
    if not exists:
        all_ok = False

assert all_ok, 'One or more required mounts are missing. Add them before continuing.'

In [ ]:
# Cell 04: Build dataset release
# Runs the full pipeline: inventory → dedupe → normalize → quality filter
# → split → benchmark-lock → caption → crops → export
from pathlib import Path
from LoRA.data.pipeline import LoRAPipeline

SOURCES_YAML = Path('/kaggle/working/VIN/LoRA/data/sources.yaml')
WORKING_DIR  = Path('/kaggle/working/vin_data')

pipeline = LoRAPipeline(config_path=SOURCES_YAML, working_dir=WORKING_DIR)
RELEASE_DIR = pipeline.run_full_pipeline()

print(f'\nRelease directory: {RELEASE_DIR}')

In [ ]:
# Cell 05: Validate release — HARD FAIL if any gate fails
from LoRA.data.validate import validate_release

result = validate_release(RELEASE_DIR)

print(f"Valid: {result['valid']}")
print(f"  train samples: {result['stats'].get('train_count', 0)}")
print(f"  val samples:   {result['stats'].get('val_count', 0)}")

if result['warnings']:
    print('\nWarnings:')
    for w in result['warnings']:
        print(f'  ⚠  {w}')

assert result['valid'], (
    'RELEASE VALIDATION FAILED:\n' + '\n'.join(f'  - {e}' for e in result['errors'])
)

In [ ]:
# Cell 06: Generate release report
from LoRA.data.report import generate_release_report
from pathlib import Path

REPORTS_DIR = Path('/kaggle/working/vin_data/reports/data') / RELEASE_DIR.name
generate_release_report(RELEASE_DIR, REPORTS_DIR)
print(f'Report saved: {REPORTS_DIR}')

In [ ]:
# Cell 07: ImageFolder contract test — HARD FAIL if columns are wrong
# Verifies that datasets.load_dataset('imagefolder', ...) produces 'image' and 'text' columns,
# which is what the trainer expects via --image_column image --caption_column text.
from LoRA.sd35_lora_training import test_imagefolder_contract

test_imagefolder_contract(RELEASE_DIR)

In [ ]:
# Cell 08: Trainer dry run — HARD FAIL if command cannot be built
# Also calls require_validated_release() to confirm status.
from LoRA.sd35_lora_training import run_lora_training

dry_run_result = run_lora_training(
    dataset_release=str(RELEASE_DIR),
    dry_run=True,
)

print('\nDry run OK. Command saved to:', dry_run_result['command_path'])

In [ ]:
# Cell 09a: Plumbing smoke train (50 steps)
# PURPOSE: Prove pipeline + adapter export/load work before investing full steps.
# Change MAX_TRAIN_STEPS in sd35_config.py LORA_TRAINING_CONFIG or override here.
import importlib, LoRA.sd35_config as _cfg

SMOKE_STEPS = 50
SMOKE_OUTPUT = '/kaggle/working/lora_smoke'

# Patch config for smoke run (does not persist to disk)
_cfg.LORA_TRAINING_MAX_TRAIN_STEPS = SMOKE_STEPS
_cfg.LORA_TRAINING_OUTPUT_DIR = __import__('pathlib').Path(SMOKE_OUTPUT)
_cfg.LORA_TRAINING_CHECKPOINTING_STEPS = SMOKE_STEPS

# Reload training module to pick up patched config
import LoRA.sd35_lora_training as _training
importlib.reload(_training)

smoke_result = _training.run_lora_training(
    dataset_release=str(RELEASE_DIR),
    dry_run=False,
)

print('\nSmoke train complete:', smoke_result)

In [ ]:
# Cell 09b: Full first experiment (1000 steps)
# Only run after smoke (Cell 09a) passes.
import importlib, LoRA.sd35_config as _cfg

_cfg.LORA_TRAINING_MAX_TRAIN_STEPS = 1000
_cfg.LORA_TRAINING_OUTPUT_DIR = __import__('pathlib').Path('/kaggle/working/sd35m-pedestrian-v1')
_cfg.LORA_TRAINING_CHECKPOINTING_STEPS = 250

import LoRA.sd35_lora_training as _training
importlib.reload(_training)

train_result = _training.run_lora_training(
    dataset_release=str(RELEASE_DIR),
    dry_run=False,
)

ADAPTER_PATH = train_result['adapter_path']
PT_PATH = train_result['pt_path']
print(f'Adapter: {ADAPTER_PATH}')
print(f'.pt:     {PT_PATH}')

In [ ]:
# Cell 09c: Review training metrics from the completed run
# Shows loss summary and curve. Fails if summary is missing (training didn't finish).
import json
from pathlib import Path

REPORTS_DIR = Path(train_result['training_reports_dir'])
summary_path = REPORTS_DIR / 'summary.json'
assert summary_path.exists(), f'Training summary missing: {summary_path}'

with open(summary_path) as f:
    summary = json.load(f)

print(f"Run ID:     {summary.get('run_id')}")
print(f"Steps:      {summary.get('total_steps')}")
print(f"Duration:   {summary.get('total_seconds', 0)/60:.1f} min")

loss = summary.get('loss', {})
print(f"Loss first: {loss.get('first')}")
print(f"Loss last:  {loss.get('last')}")
print(f"Loss min:   {loss.get('min')}")

tp = summary.get('throughput', {})
print(f"Throughput: {tp.get('steps_per_second')} steps/sec")

curve = REPORTS_DIR / 'loss_curve.png'
if curve.exists():
    from IPython.display import Image, display
    display(Image(str(curve)))


In [ ]:
# Cell 10: Export native adapter + optional .pt
# run_lora_training() already exports both; this cell re-exports if needed.
from LoRA.sd35_lora_training import export_lora_pt, verify_adapter_loadable
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/sd35m-pedestrian-v1')

verification = verify_adapter_loadable(OUTPUT_DIR / 'pytorch_lora_weights.safetensors')
print('Adapter verification:', verification)
assert verification['loadable'], f"Adapter not loadable: {verification['error']}"

In [ ]:
# Cell 11: Load-back test into sd35_model.py
# Verifies that the pipeline can attach the trained adapter via load_lora_weights().
import sys
sys.path.insert(0, '/kaggle/working/VIN')

from LoRA import sd35_model
import importlib
from pathlib import Path
import LoRA.sd35_config as _cfg

OUTPUT_DIR = Path('/kaggle/working/sd35m-pedestrian-v1')

# Point inference config to the trained adapter
_cfg.LORA_PATH = OUTPUT_DIR
_cfg.LORA_WEIGHT_NAME = 'pytorch_lora_weights.safetensors'
_cfg.LORA_ENABLED = True
_cfg.LORA_SCALE = 0.7

importlib.reload(sd35_model)

# Build and load the pipeline
pipe = sd35_model.build_sd35_pipeline()
print('Pipeline loaded with LoRA adapter.')
print('Adapter name:', _cfg.LORA_ADAPTER_NAME)

In [ ]:
# Cell 12: Fixed-prompt generation smoke test
# Generates 2 images with the trigger token to confirm the adapter is active.
import json
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/sd35m-pedestrian-v1')
VAL_PROMPTS_PATH = OUTPUT_DIR / 'validation_prompts.json'

with open(VAL_PROMPTS_PATH) as f:
    val_prompts = json.load(f)

GRID_DIR = OUTPUT_DIR / 'generated_validation_grid'
GRID_DIR.mkdir(exist_ok=True)

for i, prompt in enumerate(val_prompts['prompts'][:2]):
    print(f'Generating [{i}]: {prompt[:80]}')
    image = pipe(
        prompt=prompt,
        num_inference_steps=28,
        guidance_scale=7.0,
        generator=__import__('torch').Generator().manual_seed(42 + i),
    ).images[0]
    out_path = GRID_DIR / f'val_{i:02d}.png'
    image.save(out_path)
    print(f'  Saved: {out_path}')

print('\n✓ Generation smoke test passed')

In [ ]:
# Cell 12b: Baseline vs LoRA generation comparison
# REQUIRES: a baseline augmentation_metrics.csv from a prior run WITHOUT LoRA.
# Both runs must use the same fixed background manifest and seeds — otherwise
# the comparison is invalid.
#
# Set BASELINE_METRICS_CSV to your baseline run's metrics CSV before running.
from pathlib import Path
from LoRA.data.generation_metrics import compare_generation_runs, print_comparison_table

BASELINE_METRICS_CSV = Path('/kaggle/working/baseline_run/augmentation_metrics.csv')
LORA_METRICS_CSV     = Path('/kaggle/working/lora_run/augmentation_metrics.csv')
COMPARISON_DIR       = Path('/kaggle/working/reports/generation')

if not BASELINE_METRICS_CSV.exists():
    print(f'Skipping comparison: baseline CSV not found at {BASELINE_METRICS_CSV}')
    print('Run the augmentation pipeline without LoRA first, then rerun this cell.')
elif not LORA_METRICS_CSV.exists():
    print(f'Skipping comparison: LoRA metrics CSV not found at {LORA_METRICS_CSV}')
else:
    summary = compare_generation_runs(
        baseline_csv=BASELINE_METRICS_CSV,
        lora_csv=LORA_METRICS_CSV,
        output_dir=COMPARISON_DIR,
    )
    print_comparison_table(summary)
    print(f'\nMatched samples: {summary["matched_samples"]}')


In [ ]:
# Cell 13: Zip artifacts + provenance for download
import zipfile
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/sd35m-pedestrian-v1')
ZIP_PATH   = Path('/kaggle/working/sd35m-pedestrian-v1-artifacts.zip')

ARTIFACT_NAMES = [
    'pytorch_lora_weights.safetensors',
    'pytorch_lora_weights.pt',
    'training_config.json',
    'training_provenance.json',
    'dataset_provenance.json',
    'train_command.json',
    'pip_freeze.txt',
    'gpu_info.json',
    'validation_prompts.json',
    'adapter_verification.json',
]

with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for name in ARTIFACT_NAMES:
        p = OUTPUT_DIR / name
        if p.exists():
            zf.write(p, name)
            print(f'  + {name}')
        else:
            print(f'  - MISSING: {name}')

    # Validation grid images
    for img in sorted((OUTPUT_DIR / 'generated_validation_grid').glob('*.png')):
        zf.write(img, f'generated_validation_grid/{img.name}')
        print(f'  + generated_validation_grid/{img.name}')

    # Training metrics reports
    reports_dir = Path(train_result.get('training_reports_dir', ''))
    if reports_dir.exists():
        for f in sorted(reports_dir.iterdir()):
            zf.write(f, f'reports/training/{f.name}')
            print(f'  + reports/training/{f.name}')

print(f'\n✓ Artifacts zipped: {ZIP_PATH}')
print(f'  Size: {ZIP_PATH.stat().st_size / (1024*1024):.1f} MB')
